# Wasserstein Ascend Descend
### (C. and N.) Garcia Trillos

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import numpy as np
from Robust_nn.WAD import WAD2scale
import matplotlib.pyplot as plt
from utils.utils import read_vision_dataset
from utils.convnet import ConvNet
from utils.convnet_silu import ConvNetSiLU
import os
import torch
import gc
import torch.nn as nn
 

c:\Users\user\anaconda3\envs\min_curv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Read the DataLoader**

In [3]:
trainloader, testloader = read_vision_dataset('../data', dataset='MNIST')

100%|██████████| 9912422/9912422 [00:00<00:00, 35293293.28it/s]


Extracting ../data\MNIST\raw\train-images-idx3-ubyte.gz to ../data\MNIST\raw



100%|██████████| 28881/28881 [00:00<?, ?it/s]


Extracting ../data\MNIST\raw\train-labels-idx1-ubyte.gz to ../data\MNIST\raw



100%|██████████| 1648877/1648877 [00:00<00:00, 68423362.82it/s]


Extracting ../data\MNIST\raw\t10k-images-idx3-ubyte.gz to ../data\MNIST\raw



100%|██████████| 4542/4542 [00:00<?, ?it/s]

Extracting ../data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ../data\MNIST\raw



In [4]:
# Create some networks

n_nets = 5
net_lst = [ConvNet() for i in range(n_nets)]


In [5]:
adv_net = WAD2scale(net_list = net_lst, trainloader=trainloader, testloader = testloader, device = None , criterion= nn.CrossEntropyLoss(), scale_factor=1)
adv_net.set_optimizer()
adv_net.train(epochs = 2)


Epoch: 0
Input list shape: torch.Size([6, 128, 1, 28, 28]) Inputs shape: torch.Size([128, 1, 28, 28]) tagets shape: torch.Size([128])
Inner: 0|tensor([0.1673, 0.1668, 0.1658, 0.1656, 0.1676, 0.1669])
Inner: 1|tensor([0.1680, 0.1669, 0.1649, 0.1646, 0.1686, 0.1670])
Input list shape: torch.Size([6, 128, 1, 28, 28]) Inputs shape: torch.Size([128, 1, 28, 28]) tagets shape: torch.Size([128])
Inner: 0|tensor([0.1683, 0.1658, 0.1650, 0.1651, 0.1679, 0.1679])
Inner: 1|tensor([0.1686, 0.1645, 0.1652, 0.1656, 0.1675, 0.1687])
Input list shape: torch.Size([6, 128, 1, 28, 28]) Inputs shape: torch.Size([128, 1, 28, 28]) tagets shape: torch.Size([128])
Inner: 0|tensor([0.1697, 0.1636, 0.1643, 0.1658, 0.1689, 0.1677])
Inner: 1|tensor([0.1708, 0.1627, 0.1635, 0.1660, 0.1704, 0.1666])
Input list shape: torch.Size([6, 128, 1, 28, 28]) Inputs shape: torch.Size([128, 1, 28, 28]) tagets shape: torch.Size([128])
Inner: 0|tensor([0.1732, 0.1633, 0.1644, 0.1659, 0.1682, 0.1650])
Inner: 1|tensor([0.1754, 0.1

KeyboardInterrupt: 

In [51]:
tensor1 = torch.randn(1,10)
tensor2 = torch.randn(10,20)
torch.matmul(tensor1, tensor2).size()

torch.Size([1, 20])

In [33]:
inp,targ = next(iter(testloader))

In [35]:
net_lst[0](inp).shape

torch.Size([128, 10])

**Creating and comparing results**

In [ ]:
# options = {'only o1':(True,False, False), 'only o2':(False, True, False), 'both': (True,True, False), 'none':(False,False,False), 'modO2':(False,True,True) } 
options = {'only o1':(True,False, False), 'both': (True,True, True), 'none':(False,False,False) } 
rvec = [2,4,np.inf]
deltav = [0.2]
mdict = {}
basepath = os.curdir
mpath = os.path.join(basepath,'models')

for i,(k,(o1,o2, mod_o2)) in enumerate(options.items()):
  auxd = {}
  for r in rvec:
    rstr = str(r)
    print('\n*******\n ** Case '+k+', r=',r,'\n ******')
    torch.manual_seed(0)
    np.random.seed(0)
    auxd[rstr] = {}
    network = ConvNet()
    net_RegTrin = OrdTwoL(network, trainloader, testloader,  device='cuda', delta =0.2, r= r, o2=o2, o1=o1, mod_o2=mod_o2) #'cuda'
    net_RegTrin.set_optimizer(optim_alg='Adam', args={'lr':1e-4})
    net_RegTrin.train(epochs=5, delta=deltav)
    auxd[rstr]['train_loss'] = net_RegTrin.train_loss.copy()
    auxd[rstr]['train_acc'] = net_RegTrin.train_acc.copy()
    auxd[rstr]['train_reg'] = net_RegTrin.train_reg.copy()
    auxd[rstr]['test_acc_adv'] = net_RegTrin.test_acc_adv.copy()
    auxd[rstr]['test_acc_clean'] = net_RegTrin.test_acc_clean.copy()
    auxd[rstr]['train_times'] = net_RegTrin.train_times.copy()
    torch.save(network.state_dict(), os.path.join(mpath, k+'_r_'+str(r)+'.pth' ))
    del(network)
    gc.collect()
    torch.cuda.empty_cache()
  mdict[k] = auxd
  with open('tests_all.txt','w') as data: 
      data.write(str(mdict))